# DevIllustrate AI — Google Colab T4 GPU Stable Diffusion Runtime Verification

Before running the cells below:
1. Select **Runtime → Change runtime type → T4 GPU → Save**.
2. Upload `DevIllustrate-AI-Python-Streamlit.zip` (or the `DevIllustrate-AI/` folder) in Step 1.
3. Run cells top-to-bottom. Every timing, file size, PNG signature, and generated illustration in the final report comes strictly from live T4 GPU execution.

## Step 1: Upload & Extract `DevIllustrate-AI` and Install Required Python Dependencies

In [ ]:
import os
from pathlib import Path

if not Path('DevIllustrate-AI/requirements.txt').exists():
    from google.colab import files
    print('Upload DevIllustrate-AI-Python-Streamlit.zip ...')
    uploaded = files.upload()
    zip_name = next((k for k in uploaded.keys() if k.endswith('.zip')), None)
    assert zip_name is not None, 'Please upload a .zip containing DevIllustrate-AI/'
    !unzip -qo "{zip_name}" -d .

assert Path('DevIllustrate-AI/requirements.txt').exists(), 'DevIllustrate-AI/requirements.txt not found!'
print('Verified project directory:', sorted(os.listdir('DevIllustrate-AI')))

In [ ]:
!pip install -q --upgrade -r DevIllustrate-AI/requirements.txt

## Step 2: Optional Secure Hugging Face Authentication

- The default model **`stable-diffusion-v1-5/stable-diffusion-v1-5`** is publicly accessible and does **not** require a gated license.
- If you switch to a gated model (e.g., `stabilityai/stable-diffusion-3.5-large` or `black-forest-labs/FLUX.1-dev`), store your read token in **Colab Secrets** (left sidebar key icon → name: `HF_TOKEN`, toggle *Notebook access* ON). **Never hardcode tokens in notebook cells.**

In [ ]:
import os
from huggingface_hub import login

hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = os.getenv('HF_TOKEN')

if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    login(token=hf_token, add_to_git_credential=False)
    print('Hugging Face authentication configured securely from Colab Secrets.')
else:
    print('No HF_TOKEN found — proceeding anonymously with public default model stable-diffusion-v1-5/stable-diffusion-v1-5.')

## Step 3: Verify PyTorch, CUDA GPU, and Diffusers Installation

In [ ]:
!nvidia-smi
import torch, diffusers, transformers, accelerate, safetensors, PIL

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print('GPU Device:', torch.cuda.get_device_name(0), f'({props.total_memory / 1024**3:.1f} GB VRAM)')
else:
    print('WARNING: CUDA GPU not detected! Switch Runtime -> Change runtime type -> T4 GPU.')

print('Diffusers:', diffusers.__version__)
print('Transformers:', transformers.__version__)
print('Accelerate:', accelerate.__version__)
print('Safetensors:', safetensors.__version__)
print('Pillow:', PIL.__version__)

## Step 4: Run `verify_runtime.py` Diagnostic Check (Pre-Generation)

In [ ]:
!python DevIllustrate-AI/verify_runtime.py

## Step 5: Execute Actual Stable Diffusion Technical Illustration Generation & Save PNG

Generates a real `768×576` technical illustration using `stable-diffusion-v1-5/stable-diffusion-v1-5` (`DPMSolverMultistepScheduler`, `float16` on CUDA T4), records exact load/inference timings, verifies the PNG header & dimensions, and writes `DevIllustrate-AI/outputs/Colab_Runtime_Verification_Report.md`.

In [ ]:
TECHNICAL_PROMPT = (
    'clean professional 3D cloud computing microservices architecture diagram showing an API gateway, '
    'application load balancer, auto-scaling compute cluster, and replicated database cylinders, '
    'crisp directional data pathways, neutral off-white studio backdrop, sharp focus, zero embedded text'
)
NEGATIVE_PROMPT = (
    'blurry, low resolution, watermark, distorted text, garbled letters, messy cables, '
    'cartoon characters, noisy background, oversaturated neon'
)
OUTPUT_PNG = 'DevIllustrate-AI/outputs/colab_t4_cloud_architecture.png'
REPORT_MD = 'DevIllustrate-AI/outputs/Colab_Runtime_Verification_Report.md'

!python DevIllustrate-AI/verify_runtime.py \
  --run-generation \
  --prompt "{TECHNICAL_PROMPT}" \
  --negative-prompt "{NEGATIVE_PROMPT}" \
  --width 768 \
  --height 576 \
  --steps 30 \
  --guidance-scale 7.5 \
  --seed 4219 \
  --output "{OUTPUT_PNG}" \
  --report-md "{REPORT_MD}"

## Step 6: Verify Saved PNG Integrity & Display the Generated Illustration

In [ ]:
from pathlib import Path
from PIL import Image
from IPython.display import display, Markdown

png_path = Path('DevIllustrate-AI/outputs/colab_t4_cloud_architecture.png')
assert png_path.exists() and png_path.is_file(), f'PNG not found at {png_path}'
size_bytes = png_path.stat().st_size
assert size_bytes > 0, 'Generated PNG file is empty!'

raw_header = png_path.read_bytes()[:8]
assert raw_header == b'\x89PNG\r\n\x1a\n', f'Invalid PNG signature: {raw_header!r}'

img = Image.open(png_path)
print(f'VERIFIED PNG: {png_path} | {size_bytes:,} bytes | {img.size[0]}x{img.size[1]} | mode={img.mode}')
display(img)

## Step 7: Inspect & Download the Actual Runtime Verification Report + Generated PNG

In [ ]:
from pathlib import Path
from IPython.display import display, Markdown

report_path = Path('DevIllustrate-AI/outputs/Colab_Runtime_Verification_Report.md')
assert report_path.exists(), 'Verification report Markdown file was not generated.'
display(Markdown(report_path.read_text(encoding='utf-8')))

In [ ]:
# Optional: download both the verified PNG and the Markdown report to your local computer
from google.colab import files
files.download('DevIllustrate-AI/outputs/colab_t4_cloud_architecture.png')
files.download('DevIllustrate-AI/outputs/Colab_Runtime_Verification_Report.md')